In [ ]:
from collections import Counter
import osmnx as ox
import numpy as np
from tqdm import tqdm
import pandas as pd
from shapely import from_wkt
import threading

In [ ]:
global_hex_multi_df = pd.read_csv('global_hex_multi')
global_hex_multi_df['Multi_point'] = global_hex_multi_df['Multi_point'].apply(from_wkt)

In [ ]:
def hex_amen_counter(global_hex_multi_df,hex_id):
    hex_counter = []
    sub_trajcs = []
    for hex in global_hex_multi_df.loc[global_hex_multi_df['hex_id']==hex_id].iterrows():
        sub_trajcs.append(hex[1].values[1])
    amen = []
    amen_list = []
    for sub_traj in sub_trajcs:
        amen_sub_traj = []    
        for sub_point in sub_traj.geoms:
            try:
                amen_sub_traj.append(ox.features_from_point(sub_point.coords[0][::-1], {'amenity':True},dist=100))
            except:
                continue
        print(amen_sub_traj)
        if amen_sub_traj:
            try:
                amen.append(pd.concat(amen_sub_traj)['amenity'].values)
            except:
                amen = []
    for point_amen in amen:
        for sub_amen in point_amen:
            amen_list.append(sub_amen)
    c = Counter(amen_list)
    print(hex_id,len(sub_trajcs),c.total())
    hex_counter.append([hex_id, list(c.items())])
    hex_counter_df = pd.DataFrame(hex_counter,columns=['hex_id','amen_counter'])
    hex_counter_df.to_csv(f"./data/hex_counter/hex_{hex_id}",index=False)


In [ ]:
def hex_calc(hexes):
    t1 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[0]))
    t2 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[1]))
    t3 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[2]))
    t4 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[3]))
    t5 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[4]))
    t6 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[5]))
    t7 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[6]))
    t8 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[7]))
    t9 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[8]))
    t10 = threading.Thread(target=hex_amen_counter,args=(global_hex_multi_df,hexes[9]))
    t1.start()
    t2.start()
    t3.start()
    t4.start()
    t5.start()
    t6.start()
    t7.start()
    t8.start()
    t9.start()
    t10.start()
    t1.join()
    t2.join()
    t3.join()
    t4.join()
    t5.join()
    t6.join()
    t7.join()
    t8.join()
    t9.join()
    t10.join()

In [ ]:
for hexes in np.split(np.append(global_hex_multi_df['hex_id'].unique(),global_hex_multi_df['hex_id'].unique()[0]),15):
    # Turn on when need to recalc amenities along traj
    # hex_calc(hexes)
    continue

In [ ]:
hex_amen = []
for hex_id in global_hex_multi_df['hex_id'].unique():
    df = pd.read_csv(f"data/hex_counter/hex_{hex_id}")
    hex_amen.append(df.values)
hex_amen_df = pd.DataFrame(np.asarray(hex_amen).reshape([149,2]),columns=["hex_id","amen_along_traj"])
hex_amen_df.to_csv("Amenities_count_along_traj",index=False)